# Rossmann store sales: model evaluation and error analysis

**Goal:** understand *how well* the model forecasts, *where* it fails, and *what it relies on*, before trusting it for the 48-day test window.

Same pattern as the EDA notebook:

> **Question → Evidence → Decision**

All numbers come from **out-of-fold predictions**: each validation day is predicted by a model trained only on days before its fold (`src/validation.py`). Nothing here has seen its own answers.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # lets us `import src` from inside notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_all
from src.features import FEATURES, prepare
from src.metrics import rmspe
from src.models import GradientBoostingModel, GroupMedianBaseline
from src.validation import make_folds, split

plt.rcParams.update({
    "figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

train, test = prepare(*load_all())
folds = make_folds(train["Date"].max())

## 1. Does the model beat the baseline, consistently?

**Question:** how much better is gradient boosting than the best no-ML baseline (store × weekday × promo median), and is the gain stable across time?

This cell fits both on each of the 3 rolling folds (about 2-3 minutes) and keeps the out-of-fold predictions for the rest of the notebook.

In [ ]:
parts, scores, models = [], [], {}
for f in folds:
    tr, va = split(train, f)
    base = GroupMedianBaseline(["Store", "DayOfWeek", "Promo"]).fit(tr)
    model = GradientBoostingModel().fit(tr)
    models[f.name] = model
    va = va.assign(
        Pred=model.predict(va), BasePred=base.predict(va), Fold=f.name,
        Horizon=(va["Date"] - f.val_start).dt.days + 1,   # day 1..48 of the forecast window
    )
    parts.append(va)
    scores.append({"fold": f.name, "baseline": rmspe(va["Sales"], va["BasePred"]),
                   "model": rmspe(va["Sales"], va["Pred"])})

scores = pd.DataFrame(scores).set_index("fold")
scores["improvement"] = 1 - scores["model"] / scores["baseline"]
scores.loc["mean"] = scores.mean()
display(scores.style.format({"baseline": "{:.4f}", "model": "{:.4f}", "improvement": "{:.0%}"}))

# Scored rows only: RMSPE ignores zero-sales days.
oof = pd.concat(parts, ignore_index=True).query("Sales > 0").reset_index(drop=True)
oof["PctErr"] = (oof["Pred"] - oof["Sales"]) / oof["Sales"]   # + = over-forecast

**Finding.** The model scores about **0.106 RMSPE against 0.154** for the baseline, a **~31% reduction**, and wins on every fold (by 23-37%). The gain is not a one-window fluke.

**Decision.** Keep gradient boosting as the main model. The baseline stays in the README results table: it is what gives "0.106" its meaning.

## 2. Does error grow with the forecast horizon?

**Question:** we deliberately built no short lags (`features.py`), so day 48 of a forecast uses exactly the same information as day 1. If that design is right, error should stay roughly flat across the window. If the model were secretly relying on recent information, error would climb.

In [ ]:
by_h = oof.groupby("Horizon").apply(lambda g: rmspe(g["Sales"], g["Pred"]), include_groups=False)

fig, ax = plt.subplots()
ax.plot(by_h.index, by_h.values, marker="o", ms=3)
ax.axhline(rmspe(oof["Sales"], oof["Pred"]), ls="--", c="grey", label="overall")
ax.set(xlabel="Day of forecast window", ylabel="RMSPE", title="Error by forecast horizon (3 folds pooled)")
ax.legend(); plt.show()

bins = pd.cut(oof["Horizon"], [0, 7, 14, 28, 48], labels=["1-7", "8-14", "15-28", "29-48"])
oof.groupby(bins, observed=True).apply(lambda g: rmspe(g["Sales"], g["Pred"]), include_groups=False).round(4)

**Finding.** Error is **flat across the window**: about 0.098 in week 1 and 0.106 over days 29-48. The bumps day to day come from *which* days fall there (holidays, month-ends), not from distance.

**Decision.** This confirms the no-short-lags design. The model's quality on day 48 is the same as on day 1, which is exactly what a 48-day-ahead forecast needs. (A model with "sales yesterday" would look excellent on day 1 of validation and degrade sharply, and that degradation would be hidden if validation only looked at an average.)

## 3. Is the model biased?

**Question:** RMSPE rewards erring slightly low (`metrics.py`). Does the model systematically over- or under-forecast, overall or for particular days?

In [ ]:
def summary(g):
    return pd.Series({"rmspe": rmspe(g["Sales"], g["Pred"]), "bias": g["PctErr"].mean(), "rows": len(g)})

print(f"overall bias: {oof['PctErr'].mean():+.2%}  (mean of (pred - actual) / actual)")
oof.groupby("DayOfWeek").apply(summary, include_groups=False).style.format(
    {"rmspe": "{:.4f}", "bias": "{:+.1%}", "rows": "{:,.0f}"})

**Finding.** The model under-forecasts by about **1%** on average, and the bias is small on every weekday (within ±3%). Training on log(Sales) already leans predictions slightly low, which is what RMSPE rewards.

**Decision.** This explains why the scaling trick (multiply predictions by ~0.985) did not help in the model step: the model is already on the low side. Keep `scale = 1.0`.

## 4. Are errors spread evenly, or concentrated?

**Question:** RMSPE squares errors, so a few huge misses can dominate. Is the score a fair summary of a typical day?

In [ ]:
sq = oof["PctErr"] ** 2
top = sq.nlargest(len(oof) // 100)
print(f"worst 1% of rows produce {top.sum() / sq.sum():.0%} of total squared error")
print(f"RMSPE without them: {np.sqrt(sq.drop(top.index).mean()):.4f}  (with: {np.sqrt(sq.mean()):.4f})")
print(f"of those worst rows, over-forecasts: {(oof.loc[top.index, 'PctErr'] > 0).mean():.0%}")

per_store = oof.groupby("Store").apply(summary, include_groups=False)
fig, ax = plt.subplots()
ax.hist(per_store["rmspe"], bins=60)
ax.set(xlabel="RMSPE per store", ylabel="Stores", title="Most stores sit near 0.1; a short tail is far worse")
plt.show()
per_store.sort_values("rmspe", ascending=False).head(8).round(3)

**Finding.** The worst **1% of rows produce about 20% of the squared error**; without them RMSPE would be about 0.095. About **three-quarters (72%) are over-forecasts**: days where actual sales collapsed far below normal. Per store, the median RMSPE is about 0.096, but around 14 stores are above 0.2.

**Decision.** Investigate the worst rows before tuning anything (section 5). Tuning against a score dominated by a few anomalies would optimise for the anomalies.

## 5. What are the worst misses?

**Question:** what happened on the days the model got most wrong?

In [ ]:
cols = ["Store", "Date", "Sales", "Pred", "PctErr", "DaysUntilClosed", "StateHoliday"]
display(oof.loc[top.index[:8], cols].round(2))

# Store 292: the single worst day. Plot its last weeks before the end of training data.
s = train.query("Store == 292 and Date >= '2015-06-01'")
p = oof.query("Store == 292")
fig, ax = plt.subplots()
ax.plot(s["Date"], s["Sales"], label="actual", marker=".")
ax.plot(p["Date"], p["Pred"], label="predicted (out-of-fold)", alpha=0.8)
ax.set(title="Store 292: sales collapse the day before a 21-day closure", ylabel="Sales")
ax.legend(); fig.autofmt_xdate(); plt.show()

**Finding.** The two worst stores' worst days are the **last trading day before a long closure**: store 292 shuts for 21 days from 2015-07-11, store 909 for 16 days from 2015-07-02. These look like refurbishments; sales on the final day collapse (store 292: 1,012 actual against ~5,460 predicted). The model can't tell this apart from an ordinary "closed tomorrow" Sunday, because `DaysUntilClosed = 1` in both cases.

**Decision.** *Not* adding a "long closure ahead" feature for now. The test window contains **no long closures**: only 4 stores have closed runs, of 3-4 days. The feature would fix validation rows that don't resemble the test period. It is recorded as a **known limitation**: a production version would need it, since refurbishments do happen.

## 6. Holidays and Sundays

**Question:** do rare day types have a different error profile?

In [ ]:
display(oof.groupby("StateHoliday", observed=True).apply(summary, include_groups=False)
        .style.format({"rmspe": "{:.4f}", "bias": "{:+.1%}", "rows": "{:,.0f}"}))
display(oof.assign(Sunday=oof["DayOfWeek"] == 7).groupby("Sunday").apply(summary, include_groups=False)
        .style.format({"rmspe": "{:.4f}", "bias": "{:+.1%}", "rows": "{:,.0f}"}))

**Finding.** Open days on a state holiday are far worse (RMSPE ~0.28-0.39) and so are Sundays (~0.19), but together they are under 1% of scored rows. Few examples to learn from, and each store's holiday behaviour differs.

**Decision.** No special handling. Their effect on the overall score is small, and the test window has only 180 state-holiday rows (all type 'a'), and 154 of them are closed days, predicted as 0 by rule. Worth stating as a limitation rather than engineering around.

## 7. What does the model rely on?

**Question:** which features carry the forecast?

**Method: permutation importance** on the most recent fold. Shuffle one feature's values across rows, breaking its link to sales, and measure how much RMSPE gets worse. Unlike tree "split counts", this measures effect on the actual metric, on unseen data.

In [ ]:
f = folds[-1]
model = models[f.name]
va = oof[oof["Fold"] == f.name].reset_index(drop=True)
X = model.history_.transform(va)[FEATURES]
score = lambda X: rmspe(va["Sales"], np.expm1(model.model_.predict(X)))

base = score(X)
rng = np.random.default_rng(0)
imp = {}
for col in FEATURES:
    shuffled = [score(X.assign(**{col: X[col].values[rng.permutation(len(X))]})) for _ in range(3)]
    imp[col] = np.mean(shuffled) - base
imp = pd.Series(imp).sort_values()

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(imp.index, imp.values)
ax.set(xlabel="RMSPE increase when shuffled", title=f"Permutation importance ({f.name}, base RMSPE {base:.3f})")
plt.show()

**Finding.**
- The **store-history features dominate**: store × promo and store × weekday averages each add ~0.28 RMSPE when shuffled. The model is essentially "this store's usual level for this kind of day", then adjusted.
- **`DaysSinceClosed`** and **`DayOfMonth`** come next. Day of month is likely a payday effect (start / end of month), which the EDA didn't look at.
- **`Promo` itself scores low**, but that is a caveat of the method, not proof promo doesn't matter: its information is shared with `StorePromoMeanLogSales`, and shuffling one of two correlated features barely hurts because the other still carries the signal.
- **`Year`, `StateHoliday` and `Promo2` add nothing.** `Year` contributing nothing is good news: trees can't extrapolate to a new year, so reliance on it would be a risk.

**Decision.** Keep the feature set for now. Pruning on one fold's importance is fragile, and correlated features make individual scores misleading. If simplifying later, drop zero-importance features *and re-check RMSPE across all folds*, rather than trusting this chart alone.

## 8. Decision log

| # | Finding | Decision |
|---|---|---|
| 1 | Model 0.106 vs baseline 0.154 RMSPE (−31%), wins on every fold | Gradient boosting is the main model; baseline kept for context |
| 2 | Error flat across the 48-day window | Confirms the no-short-lags design |
| 3 | Under-forecasts by ~1% overall | Explains why prediction scaling didn't help; keep `scale = 1.0` |
| 4 | Worst 1% of rows = ~20% of squared error, 72% of them over-forecasts | Investigate anomalies before tuning |
| 5 | Worst misses = last day before long (refurbishment) closures | No new feature: test has no long closures. Documented as a limitation |
| 6 | State holidays and Sundays are poorly predicted but rare (<1% of rows) | No special handling; stated as a limitation |
| 7 | Store-history features dominate; `Year`, `StateHoliday`, `Promo2` ~zero | Keep features; any pruning must be re-validated on all folds |

**Next:** use the model for **what-if scenarios** (`scenarios.py`), e.g. "what if this store runs a promo next week?", with the EDA's caveat that the model learned associations, not proven causal effects.